# 01 — What an agent is, and when it should not be one

Module 00 was one call. The model produced text and stopped. A lot of software that is sold as an agent is still that: a completion with a nicer prompt.

This chapter names the difference that the rest of the course is built on. **Who decides the next step — you, or the model?** Once that is clear, you can say when an agent is the wrong tool, which is the sentence half this room will have to repeat to risk and compliance.


## 1. Learn

Three shapes. Same shop, same customer, different control flow.

**Chatbot**

```mermaid
flowchart LR
    A["User"] --> B["Your code"]
    B --> C["LLM"]
    C --> D["Text"]
```
**Workflow**

```mermaid
flowchart LR
    A["User"] --> B["Your code"]
    B --> C["Step 1"]
    C --> D["Step 2"]
    D --> E["Step 3"]
    E --> F["Done"]
```
**Agent**

```mermaid
flowchart LR
    A["User"] --> B["Your code"]
    B --> C["LLM"]
    C -->|"chooses"| D["Your code runs a tool"]
    D -->|"Observation"| C
    C -->|"says done, or turn cap hit"| F["Final answer"]
```

In one line each:

- **chatbot** — the model only produces text. No tools, no loop.
- **workflow** — you wrote the steps and the order. There may be no model at all.
- **agent** — the model chooses the next step. Your code runs the tools, in a loop, until you stop it.

RPA is a workflow that drives a user interface instead of an API. Same test: you wrote the steps.

An agent, in this course, is six parts, and only one of them is the model:

- a **model** that proposes the next step
- **instructions** (the system message)
- **tools** your code is willing to run
- **memory** — the list of messages, including what the tools returned
- a **loop** you wrote
- a **stop condition** — a turn cap, or the model saying it is done. Without this it runs until you pay for it

Five of the six are ordinary software. The model never runs the tool. Module 00 already showed that. Your code does.

**Shape** is what it is. **How far** is how much rope you give it. Those are different questions. We will score real processes on how far in module 17. Today there are four stops — the same four words as the slides:

- `suggest` — it recommends. A person does the thing.
- `draft` — it writes the action (the email, the refund, the ticket) and waits. A person approves or edits.
- `act with audit` — it acts. Every step is logged and reversible. A nightly export you can re-run.
- `act` — it acts. No one is in the loop. Reserve this for the lowest stakes.

A chatbot is usually `suggest`. A nightly export is usually `act with audit`. An agent that can move money should not sit on `act`.

Agents are the wrong default in three places:

1. The steps are already known. A script is cheaper, testable, and easy to explain.
2. A wrong action is expensive or hard to undo. Do not put that system on `act`.
3. The facts were never written down. A tool cannot fetch what nobody stored.

The rest of this notebook makes those three shapes visible on a two-customer shop. The real Chinook database arrives later. The two numbers we will look up are real.


## 2. Do

### Load the environment

Same load as module 00. The key is never printed.


In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing. Copy .env.example to .env and add the class key."
assert model, "MODEL_DEFAULT is missing from .env."

client = OpenAI()
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano


### A tiny shop

Two customers, two facts each: how many invoices, who the support rep is. Two functions. No model yet.


In [2]:
SHOP = {
    "helena": {"name": "Helena Holý", "invoices": 7, "rep": "Steve Johnson"},
    "puja": {"name": "Puja Srivastava", "invoices": 6, "rep": "Jane Peacock"},
}


def lookup_count(who):
    return SHOP[who]["invoices"]


def lookup_rep(who):
    return SHOP[who]["rep"]


print("helena invoices:", lookup_count("helena"), "rep:", lookup_rep("helena"))
print("puja invoices:  ", lookup_count("puja"), "rep:", lookup_rep("puja"))


helena invoices: 7 rep: Steve Johnson
puja invoices:   6 rep: Jane Peacock


### A workflow: you already know the steps

The ticket is always the same: invoice count, then the rep, then stop. You do not need a model to choose the order. You wrote the order.


In [3]:
who = "helena"
count = lookup_count(who)
rep = lookup_rep(who)
workflow_answer = f"{SHOP[who]['name']} has {count} invoices. Support rep: {rep}."
print(workflow_answer)


Helena Holý has 7 invoices. Support rep: Steve Johnson.


That is not an agent. It is also not a chatbot. It is the right design for this ticket: cheap, testable, and easy to explain to audit. Zero model calls. The number 7 came from our data.

### A chatbot: the model talks, nothing is looked up

Same question, no functions, no loop. You have seen this failure already (Prague weather). Watch it happen on a customer.


In [4]:
chat = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": "How many invoices does Helena Holý have, and who is her support rep?",
        }
    ],
    max_completion_tokens=80,
    reasoning_effort="none",
)
print(chat.choices[0].message.content)


I can’t determine that from the information provided here.  

If you share the relevant data source (e.g., a screenshot/export of the CRM/invoice table, or the system you’re using), I can tell you exactly:

- **How many invoices** Helena Holý has
- **Who her support representative** is


Whatever it said, it did not call `lookup_count` or `lookup_rep`. Compare it to the workflow cell. If the numbers match, that is luck, not a lookup.

### An agent is a loop — first with the plan written down

Before we let the model choose, look at the loop itself. These three lines are the plan. Each turn we either run a function or we stop. Ordinary Python. No magic.


In [5]:
plan = ["COUNT helena", "REP helena", "DONE"]

for step in plan:
    print("next:", step)
    if step.startswith("COUNT"):
        who = step.split()[1]
        print("saw: ", lookup_count(who))
    elif step.startswith("REP"):
        who = step.split()[1]
        print("saw: ", lookup_rep(who))
    elif step.startswith("DONE"):
        print("stop")


next: COUNT helena
saw:  7
next: REP helena
saw:  Steve Johnson
next: DONE
stop


That loop *is* the agent shape. The only thing a real agent changes: the next line is not in a list you wrote. The model emits it. Your code still runs the function. Your code still decides to stop.

### Now the model chooses the next line

Same two functions. Same three verbs: `COUNT`, `REP`, `DONE`. We will let the model pick, one line at a time, and we will send back what we saw. We cap the loop at four turns so it cannot run forever.


In [ ]:
SYSTEM = (
    "You help a music shop. You cannot see the shop data. "
    "Reply with exactly one line, nothing else:\n"
    "COUNT helena\n"
    "COUNT puja\n"
    "REP helena\n"
    "REP puja\n"
    "DONE <short answer>\n"
    "Never guess a number or a name. Look it up first. Then DONE.\n\n"
    "Example: asked about puja, your first reply is exactly:\n"
    "COUNT puja"
)

messages = [
    {"role": "system", "content": SYSTEM},
    {
        "role": "user",
        "content": "How many invoices does Helena Holý have, and who is her support rep?",
    },
]

for turn in range(4):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=64,
        reasoning_effort="none",
    )
    line = response.choices[0].message.content.strip().splitlines()[0]
    print(f"model: {line}")
    messages.append({"role": "assistant", "content": line})

    verb = line.split()[0].upper() if line.split() else ""
    who = line.split()[1].lower() if len(line.split()) > 1 else ""

    if verb == "DONE":
        break
    if verb == "COUNT" and who in SHOP:
        saw = str(lookup_count(who))
    elif verb == "REP" and who in SHOP:
        saw = lookup_rep(who)
    else:
        saw = "unknown action; use COUNT / REP / DONE"

    print(f"saw:   {saw}")
    messages.append({"role": "user", "content": "Observation: " + saw})


If the model followed the instructions, you just watched: it chose, we ran a function, we told it what we saw, it chose again. That is an agent. Module 02 will replace the home-made `COUNT helena` line with the real `tool_calls` field. Module 03 will make this loop the thing you keep.


## 3. Observe

Print the message list the live loop built. That list is the memory. Then compare the three shapes on calls and whether anyone actually used the functions.


In [7]:
for message in messages:
    role = message["role"]
    content = message["content"].replace("\n", " / ")
    print(f"{role:10} {content}")


system     You help a music shop. You cannot see the shop data. Reply with exactly one line, nothing else: / COUNT helena / COUNT puja / REP helena / REP puja / DONE <short answer> / Never guess a number or a name. Look it up first. Then DONE.
user       How many invoices does Helena Holý have, and who is her support rep?
assistant  COUNT helena
user       Observation: 7
assistant  REP helena
user       Observation: Steve Johnson
assistant  DONE Steve Johnson


In [8]:
n_model_calls = sum(1 for m in messages if m["role"] == "assistant")
print("workflow model calls:", 0)
print("chatbot model calls: ", 1)
print("agent model calls:   ", n_model_calls)
print()
print("workflow used lookup_count / lookup_rep:", True)
print("chatbot used them:                     ", False)
print("agent used them:                       ", any(
    m["role"] == "user" and m["content"].startswith("Observation:")
    for m in messages
))


workflow model calls: 0
chatbot model calls:  1
agent model calls:    3

workflow used lookup_count / lookup_rep: True
chatbot used them:                      False
agent used them:                        True


The workflow got the right answer in zero calls. The agent spent several calls to decide an order you already knew. That is not a reason to never use an agent. It is the reason this ticket should stay a workflow.

Use an agent when the next step depends on what you just found — a messy ticket, an investigation, a question you cannot outline in advance. Use a workflow when you can write the steps on a slide. Use a chatbot when talking is the whole job.

If the live model skipped the tools and guessed, that is useful too: without a loop and a tool, you are back to module 00.


## 4. Challenge

Work in twos or threes. Fill the table in this cell. There is no score.

For each job, agree on three things. The words mean the same thing they did in Learn:

- **Shape** — `chatbot` (talk only), `workflow` (you wrote the steps), or `agent` (the model chooses the next step)
- **Who decides the next step?** — you, or the model. One short phrase.
- **How far?** — `suggest`, `draft`, `act with audit`, or `act`. Same four words as the slides.

A few rows have more than one defensible answer. That is the point. Write what you would tell risk, not what you think the instructor wants.

| # | Job | Shape | Who decides the next step? | How far? |
|---|-----|-------|----------------------------|----------|
| 1 | The same overdue reminder to every account past 30 days. |  |  |  |
| 2 | A visitor asks what jazz fusion is. |  |  |  |
| 3 | A messy email with an invoice, an address change, and a question. |  |  |  |
| 4 | Auto-refund up to a set limit, straight from an inbox, no one in the loop. |  |  |  |
| 5 | Route each incoming request to billing, shipping, or product. |  |  |  |
| 6 | Why is Q3 revenue down? |  |  |  |

We will walk the table in the debrief. Disagreement on 4 and 5 is expected.
